In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
%run ./_local_config.ipynb

In [0]:
import pandas as pd
import json
from azure.storage.blob import BlobServiceClient

conn_str = f"DefaultEndpointsProtocol=https;AccountName={storage_account_name};AccountKey={storage_account_key};EndpointSuffix=core.windows.net"

blob_service = BlobServiceClient.from_connection_string(conn_str)
blob_client = blob_service.get_blob_client(container="bronze", blob="erp/battery/battery.json")

stream = blob_client.download_blob().readall()

# Decode bytes to text, split into lines, parse each line as its own JSON object
# (each line here is one full API page response, not one sales record)
text = stream.decode("utf-8-sig")
pages = [json.loads(line) for line in text.splitlines() if line.strip()]

# Each page has a "value" key containing a list of actual sales records —
# flatten all pages into a single list of records
all_records = []
for page in pages:
    all_records.extend(page["value"])

df = pd.DataFrame(all_records)

print(df.shape)
print(df.columns.tolist())
df.head()

In [0]:
print(df.shape)  # expect something in the hundreds of thousands, matching your earlier ~596K count
print(df["itemCategoryCode"].unique())

In [0]:
import sys
sys.path.append("/Workspace/Users/venura-it@brownsgroup.com/Exide sales/Exide-Sales-Forecast")

from src.transform.clean_silver import clean_to_silver

silver = clean_to_silver(df)
print(silver.shape)
silver.head()